# House Prices: reproducible model comparison and submission

This notebook starts from the original CSV files so the same learned preprocessing is applied to validation and Kaggle test data. It compares a Random Forest and an SVR, selects using cross-validated RMSLE, refits the selected pipeline on all training rows, and writes `submission.csv`.

Keep this notebook beside `train.csv` and `test.csv`, or change `DATA_DIR` below. Your original exploratory notebook is left untouched. The manual preprocessing experiments are not discarded: this notebook uses their key lesson that missing categorical values can carry meaning, while making the transformations repeatable.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import make_scorer
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR

RANDOM_STATE = 42
DATA_DIR = Path('.')

## Load and check the raw data

`Id` is retained for the submission but excluded from model features. The target is `SalePrice`; all other training columns should match the test feature columns.

In [2]:
train_path = DATA_DIR / 'train.csv'
test_path = DATA_DIR / 'test.csv'

if not train_path.exists():
    raise FileNotFoundError(f'Training file not found: {train_path.resolve()}')
if not test_path.exists():
    raise FileNotFoundError(f'Test file not found: {test_path.resolve()}')

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

required_train_columns = {'Id', 'SalePrice'}
if not required_train_columns.issubset(train_df.columns):
    raise ValueError(f'Train data must contain columns: {required_train_columns}')
if 'Id' not in test_df.columns:
    raise ValueError('Test data must contain an Id column.')
if train_df['Id'].duplicated().any() or test_df['Id'].duplicated().any():
    raise ValueError('Duplicate Id values found.')

print(f'Train shape: {train_df.shape}')
print(f'Test shape:  {test_df.shape}')
train_df.head()

Train shape: (1460, 81)
Test shape:  (1459, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## Separate features, target, and submission IDs

The Kaggle test set must have the same feature names as the training set after dropping `SalePrice` and `Id`. The check below catches column mismatches before model fitting.

In [ ]:
X = train_df.drop(columns=['SalePrice', 'Id']).copy()
y = train_df['SalePrice'].copy()
test_ids = test_df['Id'].copy()
X_kaggle_test = test_df.drop(columns=['Id']).copy()

missing_test_columns = sorted(set(X.columns) - set(X_kaggle_test.columns))
extra_test_columns = sorted(set(X_kaggle_test.columns) - set(X.columns))
if missing_test_columns or extra_test_columns:
    raise ValueError(
        'Train/test feature columns differ. '
        f'Missing from test: {missing_test_columns}; extra in test: {extra_test_columns}'
    )

X_kaggle_test = X_kaggle_test[X.columns]

if y.isna().any() or (y < 0).any():
    raise ValueError('SalePrice must be present and non-negative for RMSLE modeling.')

print(f'Feature rows/columns: {X.shape}')
print(f'Numeric features: {len(X.select_dtypes(include=np.number).columns)}')
print(f'Categorical features: {len(X.select_dtypes(exclude=np.number).columns)}')
print(f'Missing training values: {int(X.isna().sum().sum())}')

Feature rows/columns: (1460, 79)
Numeric features: 36
Categorical features: 43
Missing training values: 7829


## Reusable preprocessing and model builders

Numeric columns get median imputation and scaling; scaling is important for SVR and harmless for a Random Forest. Categorical missing values become an explicit `Missing` category, then one-hot encoding handles categories the model did not see during fitting. Each model gets its own preprocessing instance, and scikit-learn fits it only on that model's training fold.

The competition is scored with RMSLE. The target wrapper trains on `log1p(SalePrice)` and converts predictions back to dollars.

In [4]:
def make_preprocessor():
    numeric_features = X.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

    numeric_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
        ('scaler', StandardScaler()),
    ])
    categorical_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ])

    return ColumnTransformer(transformers=[
        ('numeric', numeric_pipeline, numeric_features),
        ('categorical', categorical_pipeline, categorical_features),
    ])


def make_model(regressor):
    feature_pipeline = Pipeline(steps=[
        ('preprocessor', make_preprocessor()),
        ('regressor', regressor),
    ])
    return TransformedTargetRegressor(
        regressor=feature_pipeline,
        func=np.log1p,
        inverse_func=np.expm1,
    )


def rmsle(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    y_pred = np.clip(y_pred, 0, None)
    return float(np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2)))


rmsle_scorer = make_scorer(rmsle, greater_is_better=False)

## Compare models with cross-validation

These are starting configurations, not guaranteed optimal settings. The SVR uses a linear kernel and `C=100`, reflecting the strongest SVR result saved in the earlier notebook. The Random Forest is a solid starting point. Five-fold CV compares them on the same training rows using the competition metric; lower RMSLE is better.

In [ ]:
models = {
    'Random Forest': make_model(RandomForestRegressor(
        n_estimators=500,
        max_features=1.0,
        min_samples_leaf=1,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
    'SVR (linear)': make_model(SVR(
        kernel='linear',
        C=100,
        epsilon=0.1,
    )),
}

cv_results = []
for model_name, model in models.items():
    fold_scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring=rmsle_scorer,
        n_jobs=1,
    )
    rmsle_scores = -fold_scores
    cv_results.append({
        'Model': model_name,
        'Mean CV RMSLE': rmsle_scores.mean(),
        'Std CV RMSLE': rmsle_scores.std(),
    })

cv_results_df = pd.DataFrame(cv_results).sort_values('Mean CV RMSLE').reset_index(drop=True)
cv_results_df

,Model,Mean CV RMSLE,Std CV RMSLE
0,Random Forest,0.142684,0.009235
1,SVR (linear),0.159167,0.032084


## Holdout check

Use one fixed 20% holdout split as a sanity check. This is separate from the five-fold CV above. The CV result chooses the model; the holdout gives an additional estimate of how it behaves on unseen rows.

In [6]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

holdout_results = []
for model_name, model in models.items():
    model.fit(X_train, y_train)
    valid_predictions = np.clip(model.predict(X_valid), 0, None)
    holdout_results.append({
        'Model': model_name,
        'Holdout RMSLE': rmsle(y_valid, valid_predictions),
    })

holdout_results_df = pd.DataFrame(holdout_results).sort_values('Holdout RMSLE')
holdout_results_df

,Model,Holdout RMSLE
1,SVR (linear),0.137411
0,Random Forest,0.146442


## Select, refit on all labeled rows, and predict Kaggle test rows

The selected model name comes from cross-validation, not from the Kaggle test set. Refit that model using all 1,460 labeled rows before generating the submission predictions.

In [7]:
selected_model_name = cv_results_df.loc[0, 'Model']
print(f'Selected by mean CV RMSLE: {selected_model_name}')

final_model = models[selected_model_name]
final_model.fit(X, y)
test_predictions = np.clip(final_model.predict(X_kaggle_test), 0, None)

if len(test_predictions) != len(test_ids):
    raise ValueError('Prediction count does not match the number of test rows.')
if not np.isfinite(test_predictions).all():
    raise ValueError('Predictions contain non-finite values.')

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': test_predictions,
})
submission.head()

Selected by mean CV RMSLE: Random Forest


,Id,SalePrice
0,1461,124852.442394
1,1462,154887.113773
2,1463,177005.461598
3,1464,181347.882353
4,1465,197018.206216


## Check and save the submission

The output has the same two columns and row order as the supplied sample submission. Open `submission.csv` after this cell completes and upload it to Kaggle.

In [8]:
expected_columns = ['Id', 'SalePrice']
if submission.columns.tolist() != expected_columns:
    raise ValueError(f'Unexpected submission columns: {submission.columns.tolist()}')
if submission['SalePrice'].isna().any():
    raise ValueError('Submission contains missing predictions.')

submission_path = DATA_DIR / 'submission.csv'
submission.to_csv(submission_path, index=False)
print(f'Saved {len(submission)} predictions to: {submission_path.resolve()}')
submission.describe(include='all')

Saved 1459 predictions to: D:\study\python\housePrice\submission.csv


,Id,SalePrice
count,1459.000000,1459.000000
mean,2190.000000,176437.840278
std,421.321334,71972.971740
min,1461.000000,55014.838031
25%,1825.500000,129106.344715
50%,2190.000000,156943.981229
75%,2554.500000,206859.165041
max,2919.000000,500931.785846
